# Human-in-the-Loop Workflows

Some decisions require human judgment. `GraphInterrupt` pauses the graph at any node, serializes the current state to the checkpointer, and surfaces that state to a reviewer. When the reviewer responds, `resume()` restarts execution from the exact interrupt point with their input merged into the state.

**What you'll build:** A content moderation pipeline that auto-approves safe content, auto-rejects clearly harmful content, and pauses for human review on borderline cases — surviving process restarts between the pause and the resume.

**Time:** ~25 min | **Difficulty:** Intermediate

## Prerequisites & Setup

You need an `OPENAI_API_KEY` environment variable set.

**What you'll learn:**
- Design a graph with conditional routing to three decision paths
- Raise `GraphInterrupt` inside a node to pause execution
- Handle `GraphInterruptEvent` in the caller and display state to a reviewer
- Resume a paused graph with `resume(updates={...})`
- Use the `approval_node()` shortcut for common approve/reject patterns

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key before running
# os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 1: Define the Graph State

Every field is serializable so it can be checkpointed and restored across process restarts.

In [ ]:
from __future__ import annotations
import asyncio
from dataclasses import dataclass, field
from typing import Optional

from synapsekit.graph import StateGraph, GraphInterrupt, CompiledGraph
from synapsekit.graph.checkpointing import SQLiteCheckpointer
from synapsekit.graph.visualizer import GraphVisualizer
from synapsekit.llms.openai import OpenAILLM
from synapsekit import LLMConfig

# Every field is serializable so it can be checkpointed.
@dataclass
class ModerationState:
    # Input
    content: str                        = ""
    content_id: str                     = ""

    # Set by the classify_content node
    score: float                        = 0.0   # 0.0 (harmful) to 1.0 (safe)
    classification: str                 = ""     # "safe" | "borderline" | "unsafe"
    classification_reason: str          = ""

    # Set by a human reviewer via resume(updates={...})
    human_approved: Optional[bool]      = None
    reviewer_id: str                    = ""
    reviewer_notes: str                 = ""

    # Set by the audit node
    final_decision: str                 = ""    # "approved" | "rejected" | "escalated"
    audit_log: list[str]                = field(default_factory=list)

## Step 2: Implement the Nodes

The `flag_for_review` node raises `GraphInterrupt` to pause the graph and wait for human input.

In [ ]:
import json

llm = OpenAILLM(
    model="gpt-4o-mini",
    config=LLMConfig(temperature=0.0),  # Deterministic scoring
)

async def classify_content(state: ModerationState) -> ModerationState:
    """Ask the LLM to score content on a 0\u20131 safety scale."""
    prompt = f"""Rate the following content on a safety scale from 0.0 to 1.0.
0.0 = clearly harmful (violence, hate speech, explicit content, PII exposure)
0.5 = borderline (ambiguous, context-dependent)
1.0 = clearly safe (constructive, informative, on-topic)

Respond in JSON: {{"score": <float>, "classification": "<safe|borderline|unsafe>", "reason": "<brief>"}}

Content: {state.content}"""

    response = await llm.agenerate(prompt)

    data = json.loads(response.text)
    state.score                   = float(data["score"])
    state.classification          = data["classification"]
    state.classification_reason   = data["reason"]

    print(f"[classify_content] score={state.score:.2f}  class={state.classification}")
    print(f"                   reason: {state.classification_reason}")
    return state


def auto_approve(state: ModerationState) -> ModerationState:
    """Automatically approve high-confidence safe content."""
    state.final_decision = "approved"
    state.audit_log.append(f"Auto-approved (score={state.score:.2f})")
    print(f"[auto_approve] Content {state.content_id!r} approved automatically.")
    return state


def auto_reject(state: ModerationState) -> ModerationState:
    """Automatically reject high-confidence unsafe content."""
    state.final_decision = "rejected"
    state.audit_log.append(
        f"Auto-rejected (score={state.score:.2f}, reason={state.classification_reason})"
    )
    print(f"[auto_reject] Content {state.content_id!r} rejected automatically.")
    return state


def flag_for_review(state: ModerationState) -> ModerationState:
    """Pause the graph and wait for a human reviewer.

    Raising GraphInterrupt causes the graph engine to serialize state to the
    checkpointer and suspend. The caller receives a GraphInterruptEvent with the
    current state. When they call graph.resume(), execution continues from here.
    """
    print(f"[flag_for_review] Flagging {state.content_id!r} for human review.")
    print(f"  Score: {state.score:.2f}  Reason: {state.classification_reason}")
    print("  Graph pausing \u2014 waiting for reviewer input...")

    raise GraphInterrupt(
        state=state,
        message=(
            f"Content requires human review. "
            f"Score: {state.score:.2f}. Reason: {state.classification_reason}"
        ),
        interrupt_type="human_review",
    )


def audit(state: ModerationState) -> ModerationState:
    """Log the final decision and close the moderation record."""
    entry = (
        f"decision={state.final_decision}  "
        f"score={state.score:.2f}  "
        f"reviewer={state.reviewer_id or 'auto'}  "
        f"notes={state.reviewer_notes or 'n/a'}"
    )
    state.audit_log.append(entry)
    print(f"[audit] Final decision for {state.content_id!r}: {state.final_decision}")
    print(f"        {entry}")
    return state

## Step 3: Define Routing Logic

Scores >= 0.9 auto-approve, scores < 0.4 auto-reject, and everything in between goes to human review.

In [ ]:
def route_by_score(state: ModerationState) -> str:
    """Return the name of the next node based on the classification score."""
    if state.score >= 0.9:
        return "auto_approve"      # Very safe — no human review needed
    elif state.score < 0.4:
        return "auto_reject"       # Very unsafe — no human review needed
    else:
        return "flag_for_review"   # Borderline — escalate to human

## Step 4: Build the Graph with Checkpointing

`SQLiteCheckpointer` persists state after every node. If the process crashes mid-run, `resume()` reads from this database.

In [ ]:
def build_graph() -> CompiledGraph:
    checkpointer = SQLiteCheckpointer(db_path="./moderation_checkpoints.db")

    graph = StateGraph(ModerationState, checkpointer=checkpointer)

    graph.add_node("classify_content", classify_content)
    graph.add_node("auto_approve",     auto_approve)
    graph.add_node("auto_reject",      auto_reject)
    graph.add_node("flag_for_review",  flag_for_review)
    graph.add_node("audit",            audit)

    graph.set_entry_point("classify_content")

    # After classify_content, call route_by_score to choose the next node
    graph.add_conditional_edges(
        "classify_content",
        route_by_score,
        {
            "auto_approve":    "auto_approve",
            "auto_reject":     "auto_reject",
            "flag_for_review": "flag_for_review",
        }
    )

    # All three branches converge on audit
    graph.add_edge("auto_approve",    "audit")
    graph.add_edge("auto_reject",     "audit")
    graph.add_edge("flag_for_review", "audit")  # Resumes here after GraphInterrupt

    return graph.compile()

## Step 5: Run the Graph and Handle Interrupts

`arun()` executes to completion or until a `GraphInterrupt`. The `GraphInterruptEvent` carries the current state for the reviewer.

In [ ]:
from synapsekit.graph import GraphInterruptEvent

async def moderate_content(content: str, content_id: str) -> ModerationState:
    """Run a piece of content through the moderation graph."""
    graph = build_graph()
    initial_state = ModerationState(content=content, content_id=content_id)

    try:
        # arun() executes to completion or until a GraphInterrupt
        final_state = await graph.arun(initial_state, run_id=content_id)
        print(f"\nCompleted without human review. Decision: {final_state.final_decision}")
        return final_state

    except GraphInterruptEvent as evt:
        # The graph is paused — display state to the reviewer
        print(f"\nGraph interrupted for human review.")
        print(f"  Content:  {evt.state.content[:100]}")
        print(f"  Score:    {evt.state.score:.2f}")
        print(f"  Reason:   {evt.state.classification_reason}")
        print(f"  Run ID:   {content_id}  (use this to resume)")

        # In production, send this to a review dashboard and return.
        # Here we simulate an immediate human decision.
        return await simulate_human_review(graph, content_id)


async def simulate_human_review(graph: CompiledGraph, run_id: str) -> ModerationState:
    """Simulate a reviewer approving the flagged content."""
    print("\n[Simulating reviewer decision: APPROVED]")

    # resume() restarts the graph from the interrupt point.
    # updates= merges new values into the state before execution continues.
    final_state = await graph.resume(
        run_id=run_id,
        updates={
            "human_approved": True,
            "final_decision":  "approved",
            "reviewer_id":     "mod-007",
            "reviewer_notes":  "Context is educational; not harmful.",
        }
    )
    print(f"\nResumed. Final decision: {final_state.final_decision}")
    return final_state

## Complete Working Example

Run three content samples through the moderation pipeline — one safe, one borderline, one unsafe.

In [ ]:
SAMPLES = [
    ("How to bake sourdough bread at home \u2014 step by step guide.", "post-safe-001"),
    ("My experience getting help for mental health issues \u2014 sharing my story.", "post-borderline-042"),
    ("Step-by-step instructions for bypassing security systems.", "post-unsafe-007"),
]

async def main():
    for content, cid in SAMPLES:
        print(f"\n{'='*70}")
        print(f"Moderating: {cid}")
        result = await moderate_content(content, cid)

        print(f"\nAudit log:")
        for entry in result.audit_log:
            print(f"  - {entry}")

asyncio.run(main())

## Using the `approval_node()` Shortcut

For simple approve/reject workflows, SynapseKit provides a convenience node builder that wraps `GraphInterrupt` for you.

In [ ]:
from synapsekit.graph import approval_node

# approval_node() creates a pre-built GraphInterrupt node.
# It surfaces the named field to the reviewer and waits for {"approved": True/False}.
review_node = approval_node(
    prompt_field="classification_reason",  # Display this field to the reviewer
    approved_field="human_approved",       # Write the decision here in state
    on_approve="audit",                    # Next node if approved
    on_reject="auto_reject",               # Next node if rejected
)

# Drop-in replacement for the custom flag_for_review node:
# graph.add_node("flag_for_review", review_node)
print("approval_node() created successfully — drop it in as a graph node.")

## Next Steps

- **[Checkpointing and Resumable Workflows](checkpointing-resumable.ipynb)** — deeper dive into the checkpointer API
- **[Graph Error Recovery](error-recovery.ipynb)** — handle node failures gracefully
- **[Mermaid Visualization](mermaid-visualization.ipynb)** — visualize this graph with `get_mermaid()`